In [1]:
import torch
from torch import nn
from torch.nn import functional as F
net=nn.Sequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
X=torch.rand(2,20)
net(X)

tensor([[ 0.3719, -0.0301,  0.1182,  0.0994, -0.0650, -0.1234,  0.1386, -0.0325,
         -0.0172, -0.1576],
        [ 0.1411, -0.2041,  0.2409,  0.1575, -0.1506,  0.0442,  0.2097, -0.1185,
          0.1332, -0.1518]], grad_fn=<AddmmBackward0>)

In [8]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden=nn.Linear(20,256)
        self.out=nn.Linear(256,10)
    def forward(self,X):
        return self.out(F.relu(self.hidden(X)))

In [9]:
net=MLP()
net(X)

tensor([[-0.2803,  0.0467,  0.0454,  0.1200,  0.1209,  0.0074,  0.1212, -0.0522,
         -0.0815, -0.0636],
        [-0.3299,  0.0985, -0.0253,  0.2768,  0.0916, -0.0106,  0.1338, -0.0943,
         -0.1132, -0.0666]], grad_fn=<AddmmBackward0>)

In [20]:
class MySequential(nn.Module):
    def __init__(self,*args):
        super().__init__()
        for block in args:
            self._modules[block]=block
    def forward(self,x):
        for block in self._modules.values():
            x=block(x)
        return x

In [21]:
net=MySequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
net(X)

tensor([[-0.4258, -0.1639, -0.0525, -0.0459,  0.1403,  0.1526,  0.2417,  0.1216,
          0.0843,  0.0888],
        [-0.3636, -0.1785, -0.0126,  0.0497,  0.1463,  0.2082,  0.2149,  0.0133,
          0.1460,  0.1940]], grad_fn=<AddmmBackward0>)

In [24]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight=torch.rand((20,20),requires_grad=False)
        self.linear=nn.Linear(20,20)
    def forward(self,X):
        X=self.linear(X)
        X=F.relu(torch.mm(X,self.rand_weight)+1)
        X=self.linear(X)
        while X.abs().sum()>1:
            X/=2
        return X.sum()
net=FixedHiddenMLP()
net(X)

tensor(-0.0228, grad_fn=<SumBackward0>)